In [ ]:
%matplotlib qt

In [ ]:
pip uninstall matplotlib -y

In [ ]:
pip install matplotlib==3.8.4

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.imshow(np.random.rand(5,5))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os, csv, numpy as np
from PIL import Image

# --- Paths ---
input_folder = r"C:\Users\Yash Dudhade\OneDrive\Desktop\Visualizations"
output_csv = r"C:\Users\Yash Dudhade\OneDrive\Desktop\Output\labels.csv"

# --- Parameters ---
grid_size = 8
rows = []

def label_image(img_path):
    img = np.array(Image.open(img_path))
    h, w, _ = img.shape
    cell_h, cell_w = h // grid_size, w // grid_size
    selected = set()

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img)
    ax.set_xticks(np.arange(0, w, cell_w))
    ax.set_yticks(np.arange(0, h, cell_h))
    ax.grid(True, color='white')
    ax.set_title("🟥 Click cells to toggle | Close window when done")

    def onclick(event):
        # Ignore clicks outside the image
        if event.xdata is None or event.ydata is None:
            return
        j, i = int(event.xdata // cell_w), int(event.ydata // cell_h)
        if (i, j) in selected:
            selected.remove((i, j))
        else:
            selected.add((i, j))
        redraw()

    def redraw():
        ax.clear()
        ax.imshow(img)
        ax.set_xticks(np.arange(0, w, cell_w))
        ax.set_yticks(np.arange(0, h, cell_h))
        ax.grid(True, color='white')
        ax.set_title("🟥 Click cells to toggle | Close window when done")
        # Fill selected cells in red
        for (i, j) in selected:
            rect = patches.Rectangle(
                (j * cell_w, i * cell_h), cell_w, cell_h,
                linewidth=2, edgecolor='red', facecolor='red', alpha=0.4
            )
            ax.add_patch(rect)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.show(block=True)

    # Convert selected cells to binary vector
    label_vec = np.zeros(grid_size * grid_size, dtype=int)
    for (i, j) in selected:
        label_vec[i * grid_size + j] = 1
    return label_vec


# --- Main Loop ---
for img_name in os.listdir(input_folder):
    if not img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue
    print(f"Labeling {img_name} ...")
    labels = label_image(os.path.join(input_folder, img_name))
    rows.append([img_name] + labels.tolist())

# --- Write to CSV ---
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["image_name"] + [f"cell_{i}" for i in range(1, grid_size**2 + 1)]
    writer.writerow(header)
    writer.writerows(rows)

print(f"\n✅ Labels saved to {output_csv}")